# Week 3 Exercise: Synthetic Dataset Generator
A Synthetic dataset generator that generates synthetic data with the option of choosing your open source model. 
(Originally on Collab)


In [ ]:
!pip install -q --upgrade bitsandbytes accelerate transformers==4.57.6

In [ ]:
# Imports
from google.colab import userdata
from huggingface_hub import login
from transformers import AutoTokenizer, AutoModelForCausalLM, TextStreamer, BitsAndBytesConfig
import gradio as gr
import torch
import gc

Sign in to huggingface

In [ ]:
hf_token = userdata.get('HF_TOKEN')
login(hf_token, add_to_git_credential=True)

Models

In [ ]:
LLAMA = "meta-llama/Llama-3.2-1B-Instruct"
PHI = "microsoft/Phi-4-mini-instruct"
GEMMA = "google/gemma-3-270m-it"
QWEN = "Qwen/Qwen3-4B-Instruct-2507"
DEEPSEEK = "deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B"

System prompt

In [ ]:
system_prompt = """
You are a Synthetic Data Generator.

Your task is to generate realistic synthetic datasets based on the user’s request.

The user will provide:
- The number of samples to generate
- The dataset schema or field descriptions
- The desired output format (CSV or JSON)

Your responsibilities:
1. Generate a well-structured, consistent, and realistic synthetic dataset that follows the requested schema.
2. Ensure all values are logically valid and consistent with their field types.
3. Ensure the dataset contains exactly the requested number of samples.
4. Ensure fields remain consistent across all records.
5. Generate clean, well-organized output suitable for direct use in analysis, testing, or development.

Output rules:
- Return ONLY the dataset.
- Do NOT include explanations, commentary, markdown formatting, or code blocks.
- Do NOT include headers like "Here is the data".
- If the format requested is CSV, return valid CSV with a header row.
- If the format requested is JSON, return a valid JSON array of objects.
- Ensure the output is syntactically correct and parseable.

Quality requirements:
- Data should appear realistic but must be synthetic.
- Maintain consistent formatting across all records.
- Avoid null values unless explicitly requested.
- Avoid duplicate records unless explicitly requested.

You must strictly follow the requested format and sample count.

If the schema is not clear or the sample count is missing, ask the user to clarify before generating data.
"""

Quantization Config - this allows us to load the model into memory and use less memory

In [ ]:
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4"
)

Cache model and tokeniser to avoid loading every time we generate messages


In [ ]:
model_cache = {}
tokenizer_cache = {}

In [ ]:
def get_model(model_name, quant=True):
  if model_name not in model_cache:
      tokenizer = AutoTokenizer.from_pretrained(model_name)
      tokenizer.pad_token = tokenizer.eos_token

      if quant:
          model = AutoModelForCausalLM.from_pretrained(
              model_name,
              quantization_config=quant_config
          ).to("cuda")
      else:
          model = AutoModelForCausalLM.from_pretrained(model_name).to("cuda")

      model_cache[model_name] = model
      tokenizer_cache[model_name] = tokenizer

  return model_cache[model_name], tokenizer_cache[model_name]

In [ ]:
def build_messages(message, history):
    return (
        [{"role": "system", "content": system_prompt}]
        + [{"role": h["role"], "content": h["content"]} for h in history]
        + [{"role": "user", "content": message}]
    )

def generate(model_name, message, history, quant=True, max_new_tokens=80):
  model, tokenizer = get_model(model_name, quant)
  messages = build_messages(message, history)

  input_ids = tokenizer.apply_chat_template(
      messages,
      return_tensors="pt",
      add_generation_prompt=True
  ).to("cuda")
  attention_mask = torch.ones_like(input_ids, dtype=torch.long, device="cuda")
  streamer = TextStreamer(tokenizer)
  outputs = model.generate(
      input_ids=input_ids,
      attention_mask=attention_mask,
      max_new_tokens=max_new_tokens,
      streamer=streamer
  )

  generated_tokens = outputs[0][input_ids.shape[-1]:]
  decoded_output = tokenizer.decode(generated_tokens, skip_special_tokens=True)

  # Clean up memory
  # del model
  # gc.collect()
  # torch.cuda.empty_cache()

  return decoded_output


Setup Gradio

In [ ]:
def launch_gradio():
  def chat_wrapper(message, history, model):
    return generate(model, message, history, quant=True, max_new_tokens=1000)

  gr.ChatInterface(
      fn=chat_wrapper,
      type="messages",
      additional_inputs=gr.Dropdown(
          choices=[LLAMA, PHI, GEMMA, QWEN, DEEPSEEK],
          value=LLAMA,
          label="Open source models"
      )
  ).launch(debug=True)


Launch chatbot

In [ ]:
launch_gradio()